# 공통 분석 파이프라인 — 수치 탐색에서 해석적 증명까지

이 발표의 중심 질문은 다음과 같다. **SDP의 계수 수치해를 해석적인 potential inequality와 convergence rate 증명으로 어떻게 연결할 것인가?** 이 노트북에서는 논문과 원본 Python 구현에 따라 공통 분석 절차를 정리한다. 업데이트 대입, Gram 좌표와 행렬 성분의 전개, 수치 실험 및 정리의 증명은 각 사례 노트북에서 다룬다.

발표는 다음 순서로 진행한다.

1. **공통 분석 파이프라인:** 계수 탐색 문제가 LMI와 SDP가 되는 이유
2. [**Example I · GD**](3_2_1_Figure_1_1A_GradientDescent.ipynb): Figure 1의 수치 탐색에서 Theorem 3까지
3. [**Example II · SGD**](3_2_2_Figure_2_2A_StochasticGradientDescent.ipynb): 같은 분석법에 variance를 반영하고 Figure 2와 Theorem 5로 연결
4. [**Averaging 확장**](Averaging_expansion_Theorem_6-7_Table_2.ipynb): 평균점과 oracle 평가점을 바꾸어 Theorem 6·7을 비교하고 Table 2로 종합

세 노트북의 분석을 다음 다섯 단계로 정리한다.

| 단계 | 수학적 과제 | 결과 |
| --- | --- | --- |
| **1. 알고리즘과 가정** | 업데이트식, 함수 클래스, oracle 가정 설정 | 한 단계 갱신식 및 함수·oracle 제약 |
| **2. Parameterized potential 설정** | $\phi_k^f$의 구조 및 초기·최종 potential $\phi_0^f,\phi_N^f$의 설정 | 계수 변수와 한 단계 potential inequality |
| **3. LMI/SDP로 계수 수치 탐색** | Gram 표현, interpolation 부등식, variance 제약의 LMI화 | 함수값 계수 소거 조건, LMI, SDP 수치해 |
| **4. 계수 패턴 관찰과 단순화** | 수치 계수의 $k$-의존성과 추가 제약의 영향 분석 | 명시적 계수 수열의 후보 |
| **5. 해석적 potential inequality와 convergence rate** | multiplier·계수 recurrence의 명시적 구성 | 한 단계 부등식의 증명과 convergence rate |

GD와 SGD 노트북에서는 각 알고리즘을 위의 다섯 단계에 따라 분석한다. Averaging 노트북에서는 두 평균법을 같은 절차로 각각 분석한 뒤, 함수값 상한과 convergence rate를 비교한다.

수식은 해당 절의 논문 표기를 사용하고, 구현을 설명할 때는 원본 `.py`의 변수명을 사용한다. GD·SGD의 행렬식과 두 평균법의 추가 거리항을 각각의 출처에 맞춰 구분한다.

## 1단계 · 알고리즘과 가정

$f\in\mathcal F_L$는 $L$-smooth convex function이고, $x_\star\in\arg\min f$, $f_\star=f(x_\star)$로 둔다. 분석할 알고리즘의 상태, 업데이트식, oracle 가정과 출력 성능지표를 정한다. 이 발표에서 다루는 업데이트는 반복점·평균점·oracle 출력의 선형결합으로 주어진다.

stochastic 방법에서는 논문과 원본 코드의 균등한 $n$개 outcome을 사용한다. $\mathbb E_i$는 현재 상태까지의 정보를 고정하고 이번 outcome에 대해서만 취하는 conditional expectation이며, 수치 SDP에서는 $\frac1n\sum_{i=1}^n$으로 계산한다. GD에서는 expectation과 variance 항을 생략한다.

각 SDP에서는 함수 클래스의 상수, step-size와 averaging weight를 고정한다. 이 값들이 Gram 좌표와 제약행렬을 정한다. 일반적인 oracle에 대한 해석적 부등식은 각 사례의 5단계에서 증명한다.

| 사례 노트북 | 구체화할 상태와 가정 |
| --- | --- |
| [GD](3_2_1_Figure_1_1A_GradientDescent.ipynb) | 반복점과 정확한 gradient |
| [SGD](3_2_2_Figure_2_2A_StochasticGradientDescent.ipynb) | 반복점, stochastic oracle의 unbiasedness와 variance 상한 |
| [두 평균법](Averaging_expansion_Theorem_6-7_Table_2.ipynb) | SGD 반복점과 평균점, 평균을 만드는 순서와 oracle 평가점 |

## 2단계 · Parameterized potential 설정

GD와 SGD에서는 논문 식 (2), p.6과 식 (21), p.48의 potential을 사용한다.

$$\phi_k^f=
a_k\|x_k-x_\star\|^2+b_k\|f'(x_k)\|^2
+2c_k\langle f'(x_k),x_k-x_\star\rangle
+d_k(f(x_k)-f_\star).$$

식 (21)의 $Q_k\in\mathbb S^2$는 이차항의 계수행렬 $\begin{pmatrix}a_k&c_k\\c_k&b_k\end{pmatrix}$이다. 원본 코드에서는 수식의 상태 $k$에 해당하는 행렬과 함수값 계수를 `Q[k+1]`, `d[k+1]`에 저장한다. 중간 $Q_k$에는 대칭성을 요구하며 별도의 PSD 조건은 두지 않는다.

두 평균법의 potential은 원본 B·C 코드의 주석과 연산을 따른다. SGD with averaging에서는

$$\begin{aligned}
\phi_k^f={}&a_k\|z_k-x_\star\|^2+b_k\|f'(z_k)\|^2
+2c_k\langle f'(z_k),z_k-x_\star\rangle\\
&+d_k(f(z_k)-f_\star)+a'_k\|x_k-x_\star\|^2,
\end{aligned}$$

primal averaging에서는

$$\begin{aligned}
\phi_k^f={}&a_k\|y_k-x_\star\|^2+b_k\|f'(y_k)\|^2
+2c_k\langle f'(y_k),y_k-x_\star\rangle\\
&+d_k(f(y_k)-f_\star)+a'_k\|x_k-x_\star\|^2.
\end{aligned}$$

추가 거리항의 계수 $a'_k$는 원본 코드의 `ap[k+1]`에 해당한다. 알고리즘과 초기·최종 potential 설정으로 정한 계수 외에 나머지 계수들을 다음 한 단계 부등식을 만족하도록 찾는다.

$$\boxed{\mathbb E_i\phi_{k+1}^f\le\phi_k^f+e_k\sigma^2.}$$

GD에서는 expectation을 생략하고 $e_k=0$으로 둔다. 초기·최종 계수 조건과 중간 계수의 제한은 원본 코드의 `relax` 설정을 따른다.

### 사례별 초기·최종 potential 설정 요약

아래 표는 원본 GD·SGD·두 평균법 코드의 `relax`별 **최종 계수 조건을 대입한 $\phi_N^f$**이다. 같은 최종 형태를 갖는 설정은 한 행에 묶었다.

| 원본 구현의 사례 | `relax` | 최종 계수 조건에 따른 $\phi_N^f$ | 유도할 상한 |
| --- | --- | --- | --- |
| GD | 0, 3 | $b_N\|f'(x_N)\|^2$ | gradient norm 제곱 |
| GD | 1 | $b_N\|f'(x_N)\|^2+(2N+1)L(f(x_N)-f_\star)$ | gradient norm 제곱·함수값 오차 |
| GD | 2 | $L^2\|x_N-x_\star\|^2+b_N\|f'(x_N)\|^2+(2N+1)L(f(x_N)-f_\star)$ | gradient norm 제곱·함수값 오차 |
| SGD | 0, 1, 2 | $d_N(f(x_N)-f_\star)$ | expected 함수값 오차 |
| SGD | 3 | $d_N(f(x_N)-f_\star)+\frac L2\|x_N-x_\star\|^2$ | expected 함수값 오차 |
| SGD with averaging | 0 | $d_N(f(z_N)-f_\star)$ | 평균점의 expected 함수값 오차 |
| SGD with averaging | 1 | $d_N(f(z_N)-f_\star)+\frac L2\|x_N-x_\star\|^2$ | 평균점의 expected 함수값 오차 |
| Primal averaging | 0 | $d_N(f(y_N)-f_\star)$ | 평균점의 expected 함수값 오차 |
| Primal averaging | 1 | $d_N(f(y_N)-f_\star)+\frac L2\|x_N-x_\star\|^2$ | 평균점의 expected 함수값 오차 |

표의 $d_k$는 각 원본 코드의 `d[k+1]`에 대응하는 **potential의 함수값 계수**다. Theorem 6의 averaging weight $d_k$와는 역할이 다르며, 정리의 계수 선택은 averaging 노트북의 해석적 증명에서 따로 다룬다. 같은 $\phi_N^f$ 형태를 갖더라도 중간 계수의 제약이나 초기 potential이 다르면 서로 다른 SDP다. 예를 들어 GD의 `relax=0,3`은 최종 potential이 같지만, `relax=3`은 중간 함수값 계수도 $d_k=0$으로 고정한다.

이제 potential의 형태와 초기·최종 계수를 정했으므로, 한 단계 부등식을 LMI로 나타낸다.

## 3단계 · LMI/SDP로 계수 수치 탐색

### 한 단계의 Gram 표현

논문 Appendix D.4, pp.35–36과 Appendix E.7, pp.48–49에서는 필요한 실제 벡터를 $P$의 열에, 함수값을 행벡터 $F$에 모은다. 일반성을 잃지 않고 $x_\star=f_\star=0$으로 평행이동한 좌표를 사용한다.

$$G=P^\top P\succeq0.$$

GD에서는 $I_k=\{k,k+1,\star\}$의 각 점에 대해 논문의 계수벡터가

$$P\mathbf x_i=x_i,\qquad
P\mathbf g_i=f'(x_i),\qquad
F\mathbf f_i=f(x_i),\qquad
\mathbf x_\star=\mathbf g_\star=\mathbf f_\star=0$$

를 만족한다. 여기서 $i$는 이 집합의 함수 평가점 번호다.

SGD에서는 논문 E.7에 따라 현재 점을 $0$, outcome별 다음 점을 $1,\ldots,n$으로 번호 붙여 $I=\{\star,0,1,\ldots,n\}$을 사용한다. 계수벡터의 관계는

$$\begin{aligned}
P\mathbf x_0&=x_k,&P\mathbf x_i&=x_{k+1}^{(i)},\\
P\mathbf g_0&=f'(x_k),&P\mathbf g_i&=f'(x_{k+1}^{(i)}),\\
F\mathbf f_0&=f(x_k),&F\mathbf f_i&=f(x_{k+1}^{(i)})
\end{aligned}$$

이다. 업데이트식과 oracle의 unbiasedness도 논문의 다음 계수벡터 관계에 반영한다.

$$P\mathbf G_i=G(x_k;i),\qquad
\mathbf g_0=\frac1n\sum_{i=1}^n\mathbf G_i,\qquad
\mathbf x_i=\mathbf x_0-\delta_k\mathbf G_i.$$

$P,F$의 열 순서와 각 계수벡터의 성분은 사례 노트북에서 전개한다. 원본 코드의 `statesKG`, `statesK1G`는 potential에 필요한 위치·gradient의 **계수벡터를 열로 모은 배열**이다. $G$는 Gram 행렬이고, 원본 코드의 oracle $G(x;i)$와는 역할이 다르다. 코드의 딕셔너리 `G`, `F` 역시 수식의 Gram 행렬과 함수값 행벡터에 해당하는 변수가 아니므로, 코드 이름은 백틱으로 구분한다.

### 논문의 potential 차이 표현

GD의 Appendix D.4, p.36에서는 다음 행렬을 사용한다.

$$\begin{aligned}
V_k&=[\mathbf x_k-\mathbf x_\star\quad\mathbf g_k]
\begin{pmatrix}a_k&c_k\\c_k&b_k\end{pmatrix}
[\mathbf x_k-\mathbf x_\star\quad\mathbf g_k]^\top,\\
V_{k+1}&=[\mathbf x_{k+1}-\mathbf x_\star\quad\mathbf g_{k+1}]
\begin{pmatrix}a_{k+1}&c_{k+1}\\c_{k+1}&b_{k+1}\end{pmatrix}
[\mathbf x_{k+1}-\mathbf x_\star\quad\mathbf g_{k+1}]^\top.
\end{aligned}$$

따라서

$$\boxed{\begin{aligned}
\phi_{k+1}^f-\phi_k^f
={}&\operatorname{Tr}\!\bigl(G(V_{k+1}-V_k)\bigr)\\
&+F\bigl[d_{k+1}(\mathbf f_{k+1}-\mathbf f_\star)-d_k(\mathbf f_k-\mathbf f_\star)\bigr].
\end{aligned}}$$

SGD의 Appendix E.7, p.49에서는 현재 점과 outcome별 다음 점에 대해

$$\begin{aligned}
V_k&=[\mathbf x_0-\mathbf x_\star\quad\mathbf g_0]Q_k
[\mathbf x_0-\mathbf x_\star\quad\mathbf g_0]^\top,\\
V_{k+1}^{(i)}&=[\mathbf x_i-\mathbf x_\star\quad\mathbf g_i]Q_{k+1}
[\mathbf x_i-\mathbf x_\star\quad\mathbf g_i]^\top
\end{aligned}$$

로 정의하므로

$$\boxed{\begin{aligned}
\mathbb E_i\phi_{k+1}^f-\phi_k^f
={}&\operatorname{Tr}\!\left[G\left(\frac1n\sum_{i=1}^nV_{k+1}^{(i)}-V_k\right)\right]\\
&+F\left[\frac{d_{k+1}}n\sum_{i=1}^n(\mathbf f_i-\mathbf f_\star)
-d_k(\mathbf f_0-\mathbf f_\star)\right].
\end{aligned}}$$

Potential 계수를 고정하면 위 차이는 $G,F$에 선형이다. 업데이트를 고정하면 $V_k,V_{k+1},V_{k+1}^{(i)}$의 성분은 potential 계수에 선형이다. 두 성질은 각각 나머지 대상을 고정했을 때 성립한다.

평균법에는 2단계에서 쓴 추가 거리항이 있으므로 각 원본 코드의 조립식을 따른다. B·C 코드의 현재 추가 거리항은 `ap[k] * np.outer(xk-xs, xk-xs)`이고, 다음 추가 거리항은 `ap[k+1] * np.outer(XK1[i-1]-xs, XK1[i-1]-xs)`를 outcome별로 평균 낸다. 이 코드 식의 `k`는 원본 루프와 같이 1부터 시작한다. 구체적인 행렬 성분의 전개는 각 사례의 3단계에서 다룬다.

### 함수 클래스와 oracle 제약

$\mathcal F_L$의 smooth convex interpolation 부등식은

$$f(x_i)-f(x_j)-\langle f'(x_j),x_i-x_j\rangle
-\frac1{2L}\|f'(x_i)-f'(x_j)\|^2\ge0$$

이다. Appendix D.4의 식 (19), p.35에서 사용하는 행렬은

$$\begin{aligned}
M&=\frac12\begin{pmatrix}
0&0&0&-1\\0&0&0&1\\0&0&-L^{-1}&L^{-1}\\-1&1&L^{-1}&-L^{-1}
\end{pmatrix},\\
M_{i,j}&=[\mathbf x_i\ \mathbf x_j\ \mathbf g_i\ \mathbf g_j]
M[\mathbf x_i\ \mathbf x_j\ \mathbf g_i\ \mathbf g_j]^\top
\end{aligned}$$

이므로, 같은 부등식을 다음처럼 쓸 수 있다.

$$\boxed{F(\mathbf f_i-\mathbf f_j)+\operatorname{Tr}(GM_{i,j})\ge0.}$$

각 점·gradient의 계수벡터와 $L$을 고정하면 $M_{i,j}$는 상수행렬이다. $i=j$인 항은 0이며, 원본 코드는 서로 다른 순서쌍에만 multiplier를 곱한다.

SGD의 Appendix E.7, p.49에서는 variance 행렬을

$$A_{\rm var}=\frac1n\sum_{i=1}^n
(\mathbf G_i-\mathbf g_0)(\mathbf G_i-\mathbf g_0)^\top$$

로 정의한다. 따라서 variance 상한은

$$\operatorname{Tr}(GA_{\rm var})
=\mathbb E_i\|G(x_k;i)-f'(x_k)\|^2\le\sigma^2$$

가 된다. 원본 SGD 코드의 `Avar`가 이 행렬에 해당한다. 두 평균법은 각각의 oracle 평가점에서 variance를 계산하여 원본 B·C 코드의 `Avar`에 저장한다. GD에서는 variance 제약을 생략한다.

### 함수값 계수 소거와 LMI

GD의 Appendix D.4, p.36 하단 박스는 $I_k=\{k,k+1,\star\}$에 대해 다음 조건을 제시한다.

$$\boxed{\begin{gathered}
V_{k+1}-V_k+\sum_{i,j\in I_k}\lambda_{i,j}M_{i,j}\preceq0,\\
d_{k+1}(\mathbf f_{k+1}-\mathbf f_\star)-d_k(\mathbf f_k-\mathbf f_\star)
+\sum_{i,j\in I_k}\lambda_{i,j}(\mathbf f_i-\mathbf f_j)=0,\\
\lambda_{i,j}\ge0.
\end{gathered}}$$

SGD의 Appendix E.7, p.49 하단 박스는 $I=\{\star,0,1,\ldots,n\}$에 대해 다음 조건을 제시한다.

$$\boxed{\begin{gathered}
\frac1n\sum_{i=1}^nV_{k+1}^{(i)}-V_k-e_kA_{\rm var}
+\sum_{i,j\in I}\lambda_{i,j}M_{i,j}\preceq0,\\
\frac{d_{k+1}}n\sum_{i=1}^n(\mathbf f_i-\mathbf f_\star)
-d_k(\mathbf f_0-\mathbf f_\star)
+\sum_{i,j\in I}\lambda_{i,j}(\mathbf f_i-\mathbf f_j)=0,\\
\lambda_{i,j}\ge0,\qquad e_k\ge0.
\end{gathered}}$$

이 조건은 potential 차이에 interpolation 부등식과 variance 제약의 비음수 가중합을 더해서 얻는다. SGD의 경우 더하는 식은

$$\sum_{i,j\in I}\lambda_{i,j}\bigl[F(\mathbf f_i-\mathbf f_j)+\operatorname{Tr}(GM_{i,j})\bigr]
+e_k\bigl(\sigma^2-\operatorname{Tr}(GA_{\rm var})\bigr)$$

이다. 이를 $\mathbb E_i\phi_{k+1}^f-\phi_k^f-e_k\sigma^2$에 더하면 $e_k\sigma^2$가 소거된다. 함수값 행벡터 $F$의 계수는 위 선형 등식으로 0이 되고, 남은 항은 $G\succeq0$와 위 NSD 행렬의 trace이므로 0 이하가 된다. 실제 함수·oracle 제약을 만족하는 데이터에서는 더한 가중합이 비음수이므로 원하는 potential 부등식을 얻는다.

업데이트를 고정하면 $M_{i,j},A_{\rm var}$는 상수행렬이고, $V_k,V_{k+1},V_{k+1}^{(i)}$는 potential 계수에 선형이다. 따라서 행렬 조건의 성분과 함수값 소거식은 potential 계수와 multiplier에 대해 affine하다. 이 NSD 조건이 **Linear Matrix Inequality (LMI)**다. 각 단계의 LMI·선형 등식·부호 조건과 선형 objective를 모으면 **SDP**가 된다.

원본 구현은 행렬의 좌변을 `cons_SDP[k]`, 함수값 계수를 `cons_LIN[k]`에 조립한다. 평균법의 추가 거리항도 같은 배열에 더한다. `cons_SDP[k] << 0`, `cons_LIN[k] == 0`, `lam[k] >= 0`을 부과하고, stochastic 방법은 `e[k] >= 0`도 부과한다. 이 코드 식의 `k`는 원본 루프와 같이 1부터 시작한다.

구체적인 Gram 좌표와 행렬 성분은 [GD의 3단계](3_2_1_Figure_1_1A_GradientDescent.ipynb), [SGD의 3단계](3_2_2_Figure_2_2A_StochasticGradientDescent.ipynb), [두 평균법의 각 3단계](Averaging_expansion_Theorem_6-7_Table_2.ipynb)에서 전개한다.

### SDP objective

objective는 2단계에서 정한 최종 상한에 맞춘다. 아래 표의 계수는 원본 GD·SGD·두 평균법 코드의 `Q`, `d`, `e`에 대응한다. B 코드의 함수값 계수 $d_k$는 Theorem 6의 averaging weight $d_k$와 역할이 다르며, 코드 설명과 정리의 증명에서 각각 원래 정의를 사용한다.

| 사례 | SDP의 objective | 최종 상한에서의 의미 |
| --- | --- | --- |
| GD | $b_N$ 최대화 | gradient norm 제곱 상한의 분모를 키움 |
| SGD | 먼저 $d_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 초기 거리 항과 variance 항에 곱해지는 계수를 차례로 줄임 |
| SGD with averaging | 먼저 $d_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 평균점의 함수값 상한에서 초기 거리 항과 variance 항의 계수를 차례로 줄임 |
| Primal averaging | 정해진 $d_k$ 아래에서 $\sum e_k$ 최소화 | $d_{k+1}=d_k+L\delta_k$가 평균 업데이트에 이미 들어감 |

이처럼 objective는 각 사례에서 구하려는 상한에 따라 정한다. Primal averaging에서는 함수값 계수가 알고리즘으로 정해지므로, 단계별 오차 계수의 합을 최소화하는 SDP를 한 번 푼다.

SDP로 얻은 $a_k,b_k,c_k,d_k,e_k$와 평균법의 $a'_k$ 수열에서 $k$에 따른 패턴을 분석한다. Primal averaging의 $d_k$처럼 원본 코드가 recurrence로 고정한 계수는 decision variable이 아니다.

## 4단계 · 계수 패턴 관찰과 단순화

수치해에서 계수의 변화 양상을 살펴보고, potential을 단순화할 조건을 정한다. 그 조건을 적용한 SDP를 다시 풀어, 최종 상한이 얼마나 달라지는지 비교한다.

| 사례 | 계수 수열의 비교 항목 | 단순화한 $\phi_k^f$ |
| --- | --- | --- |
| GD | 최종 $b_N$의 증가, 함수값 계수의 선형 형태, 교차항·거리항의 계수를 제한했을 때의 변화 | 거리항 계수를 고정하고 교차항을 없앤 potential |
| SGD | $d_N$과 오차 계수의 누적합 사이의 관계, 교차항·gradient norm 제곱 항을 없애고 거리항을 고정했을 때의 변화 | $\phi_k^f=(L/2)\|x_k-x_\star\|^2+d_k(f(x_k)-f_\star)$ |
| 두 평균법 | 평균점의 이차항을 없앤 경우, 최종 거리항의 선택, 함수값 계수와 단계별 오차 계수의 관계 | 평균점의 함수값과 $x_k$의 거리항으로 이루어진 potential |

계수를 제한하는 실험에는 초기·최종 조건이 함께 바뀌는 경우도 있다. 각 노트북은 그 차이를 밝힌 상태에서 수치해를 비교한다. stochastic 방법에서는 함수값 계수뿐 아니라 **오차 계수의 누적합을 함수값 계수로 나눈 비율**까지 비교한다.

다음 단계에서는 계수 recurrence와 multiplier를 명시적으로 구성해 $\mathbb E_i\phi_{k+1}^f\le\phi_k^f+e_k\sigma^2$를 증명한다.

## 5단계 · 해석적 potential inequality와 convergence rate

단순화한 potential의 계수를 recurrence나 명시적 수열로 정하고, 사용할 interpolation 부등식과 비음수 multiplier를 선택한다. 가중합을 전개했을 때 함수값 항이 소거되거나 potential 차이로 모이고, 남은 제곱항의 부호가 정해지는 이유를 증명한다.

3단계에서는 조건을 만족하는 계수를 수치적으로 구했다. 여기서는 multiplier를 명시적인 식으로 제시하고, 함수값 항의 소거 조건과 논문의 행렬 부등식이 각 반복 단계에서 성립함을 증명한다.

| 정리 | 5단계에서 증명할 내용 |
| --- | --- |
| Theorem 3 | 세 부등식의 가중합을 GD potential의 차이로 만드는 계수 recurrence |
| Theorem 5 | conditional expectation을 취한 SGD potential의 변화량을 variance 항으로 제한하는 과정 |
| Theorem 6 | 평균 업데이트 식을 대입하고 완전제곱식으로 정리해 오차 계수를 구하는 과정 |
| Theorem 7 | 평균점에서 oracle을 평가할 때 교차항이 소거되어 더 단순한 오차 계수를 얻는 과정 |

한 단계 부등식을 증명한 뒤에는 expectation을 취하고 합하여

$$\mathbb E\phi_N^f
\le\phi_0^f+\sigma^2\sum_{k=0}^{N-1}e_k$$

를 얻는다. 최종 상한은 2단계의 원본 계수 조건에 직접 대입해서 구한다.

GD의 원본 초기조건은 $a_0=L^2$, $b_0=c_0=0$이고, $d_0$는 `relax=0,3`에서 0, `relax=1,2`에서 $L$이다. 최종 potential의 함수값·거리항은 비음수이므로 $b_N>0$일 때

$$\boxed{\|f'(x_N)\|^2
\le\frac{L^2\|x_0-x_\star\|^2+d_0(f(x_0)-f_\star)}{b_N}.}$$

SGD와 두 평균법의 원본 초기 potential은 모두 $\frac L2\|x_0-x_\star\|^2$이다. 2단계 표의 최종 거리항이 있는 설정에서는 그 비음수 항을 제외한다. 각 원본 코드의 함수값 계수 $d_N>0$에 대해 다음 상한을 얻는다.

| 원본 구현의 사례 | 함수값 오차의 상한 |
| --- | --- |
| SGD | $\displaystyle\mathbb E[f(x_N)-f_\star]\le\frac{L\|x_0-x_\star\|^2}{2d_N}+\frac{\sum_{k=0}^{N-1}e_k}{d_N}\sigma^2$ |
| SGD with averaging | $\displaystyle\mathbb E[f(z_N)-f_\star]\le\frac{L\|x_0-x_\star\|^2}{2d_N}+\frac{\sum_{k=0}^{N-1}e_k}{d_N}\sigma^2$ |
| Primal averaging | $\displaystyle\mathbb E[f(y_N)-f_\star]\le\frac{L\|x_0-x_\star\|^2}{2d_N}+\frac{\sum_{k=0}^{N-1}e_k}{d_N}\sigma^2$ |

Primal averaging에서는 원본 코드가 $d_0=0$, $d_{k+1}=d_k+L\delta_k$로 이 분모를 정한다. 각 사례의 증명에서 확인한 $b_N,d_N,e_k$ 수열을 대입하고, $N$에 따른 상한의 차수와 step-size의 영향을 평가해 convergence rate를 얻는다.

## 발표 전체를 잇는 질문

사례별 확장은 다음과 같다.

- **GD → SGD:** LMI를 구성할 때 conditional expectation과 variance 조건을 반영하면, 어떤 오차 계수가 필요하고 objective는 어떻게 달라지는가?
- **SGD → Averaging:** 함수 클래스와 oracle 가정을 유지하면서 평균점을 출력하면, 최종 함수값 상한의 오차항을 얼마나 줄일 수 있는가?
- **Averaging → Primal averaging:** 평균을 만드는 순서와 oracle 평가점을 바꾸면, 같은 분석법으로 구한 오차 계수는 어떻게 달라지는가?
- **두 평균법 → Table 2:** 앞에서 구한 명시적 계수들을 합했을 때, step-size 감소 속도는 초기 거리 항과 variance 항의 균형을 어떻게 바꾸는가?

Table 2 비교에서는 앞에서 구한 계수의 유한 합과 점근적 차수를 분석한다. 따라서 averaging 노트북의 마지막 부분에서는 두 평균법의 분석 결과를 앞선 SGD 결과와 함께 비교한다.

먼저 [**Example I · Gradient descent**](3_2_1_Figure_1_1A_GradientDescent.ipynb)에서 potential 계수 탐색부터 해석적 증명까지 전개한다.

---

참조 범위: 논문 §3.2.1·§3.2.2, Figure 1·2, Theorem 3·5·6·7 및 Table 2. 각 사례 노트북에 해당 수식·참조 코드·저장된 수치 결과와 상세 증명을 배치했다.

이 노트북의 수식과 구현 표기의 출처는 다음과 같다.

| 내용 | 논문·원본 구현 |
| --- | --- |
| GD·SGD potential과 $Q_k$ | 논문 식 (2), p.6 및 식 (21), p.48 |
| GD의 $P,F,G,V_k,V_{k+1},M_{i,j},I_k$와 LMI | Appendix D.4, pp.35–36; [원본 GD 코드](../1_Deterministic_SmoothConvex/A_GradientDescent.py) |
| SGD의 $V_{k+1}^{(i)},A_{\rm var},I$와 LMI | Appendix E.7, pp.48–49; [원본 SGD 코드](../2_Stochastic_BoundedVariance/A_StochasticGradientDescent.py) |
| Averaging potential과 `ap`, `cons_SDP`, `cons_LIN` | [원본 B 코드](../2_Stochastic_BoundedVariance/B_StochasticGradientDescentWithAveraging.py), [원본 C 코드](../2_Stochastic_BoundedVariance/C_StochasticGradientDescentWithPrimalAveraging.py) |